In [0]:
%pip install langchain langchain-openai databricks-langchain langchain-community langchain-databricks langgraph langchain_chroma
 
dbutils.library.restartPython()

In [0]:
import os
os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY

In [0]:
from langchain_openai import ChatOpenAI


llm=ChatOpenAI(model='gpt-4o')


question=input("enter the prompt")

response=llm.invoke(question)
print(response.content)

In [0]:
from langchain_openai import ChatOpenAI


llm=ChatOpenAI(model='gpt-5.4')


question=input("enter the prompt")

response=llm.invoke(question)
print(response.content)

In [0]:
from databricks_langchain import ChatDatabricks


llm=ChatDatabricks(model="databricks-claude-opus-4-1")


question=input("enter the prompt")

response=llm.invoke(question)
print(response.content)

In [0]:
import mlflow

mlflow.langchain.autolog()

from databricks_langchain import ChatDatabricks


llm=ChatDatabricks(model="databricks-claude-opus-4-1")


question=input("enter the prompt")

response=llm.invoke(question)
print(response.content)

In [0]:
from langchain.prompts import PromptTemplate
from databricks_langchain import ChatDatabricks


llm = ChatDatabricks(model="databricks-gemini-2-5-flash")

prompt_template = PromptTemplate(
    input_variables=[
        "city",
        "month",
        "language",
        "budget"
    ],
    template=(
        "Welcome to the {city} travel guide!\n"
        "If you're visiting in {month}, here is a detailed guide to help you make the most of your trip:\n"
        "Start by exploring the must-visit attractions that showcase the unique culture and history of {city}. "
        "Indulge in the local cuisine, with recommendations for dishes and restaurants that fit your taste and budget. "
        "Learn a few useful phrases in {language} to enhance your travel experience and connect with locals. "
        "Finally, discover practical tips for traveling on a {budget} budget, including affordable accommodation, transportation, and activities.\n"
        "Please provide the information in a single, well-structured paragraph."
        "Give me the output in this Kanada language"
    )
)
print("Travel Guide App")

city=input("enter your travel city")
month=input("enter your travel month")
language=input("enter your language")
budget=input("enter your budget")


if city:
    response=llm.invoke(prompt_template.format(city=city,month=month,language=language,budget=budget))
    print(response.content)

In [0]:
from databricks_langchain import ChatDatabricks
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_databricks import DatabricksEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

In [0]:

#data preprocessing
document=TextLoader("/Volumes/ai/naval/raw/product-data (2).txt").load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
chunks=text_splitter.split_documents(documents=document)

#embeddings 
embeddings=DatabricksEmbeddings(endpoint="databricks-bge-large-en")

# vectore store 
vectore_store=Chroma.from_documents(chunks,embeddings)

#serach or retriever
retriever=vectore_store.as_retriever()

#llm as brain
llm = ChatDatabricks(model="databricks-gpt-oss-20b")

prompt_template = PromptTemplate(
    input_variables=["input", "context"],
    template="""You are an assistant for answering questions.
Use the provided context to respond. If the answer isn't clear, acknowledge that you don't know. Limit your response to three concise sentences.
 
Context: {context}
Question: {input}
"""
)

qa_chain=create_stuff_documents_chain(llm,prompt_template)
rag_chain=create_retrieval_chain(retriever,qa_chain)


print("Chat with your own Data")

question=input("what is your query?")

if question:
    response=rag_chain.invoke({"input":question})
    print(response['answer'])


Foodly App

In [0]:
file_path="/Volumes/ai/naval/rag/raw/foodly_company_documents.txt"

# Read the entire text
with open(file_path, "r") as f:
    policy_text = f.read()
    print(policy_text)

In [0]:
from langchain_community.document_loaders import TextLoader

# Load the document
loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()

print(f"Loaded {len(documents)} document(s).")

print(documents[0].metadata)
#print(documents[0].page_content[:1000])

In [0]:
from langchain_community.document_loaders import TextLoader

# Load the document
loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()

print(f"Loaded {len(documents)} document(s).")

print(documents[0].metadata)
#print(documents[0].page_content[:1000])

In [0]:

from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # ~800 characters per chunk
    chunk_overlap=100,   # 100 characters overlap to preserve context
    separators=["\n\n", "\n", " ", ""]

)

docs=text_splitter.split_documents(documents)

print(f"Created {len(docs)} chunks.")



for _doc in docs:
    print(_doc)
    print("-"*100)


In [0]:

# Prepare data for Delta
chunk_data = []
for i, d in enumerate(docs):
    chunk_data.append({
        "chunk_id": i + 1,
        "content": d.page_content,
        "metadata": str(d.metadata)  # store metadata as JSON-like string
    })


In [0]:
chunk_data

In [0]:
df=spark.createDataFrame(chunk_data)

In [0]:
display(df)

In [0]:
df.write.mode("overwrite").saveAsTable("ai.naval.foodly_chuck_docs")

In [0]:
%sql
use catalog ai;
create schema if not exists ai.naval;

In [0]:
spark.sql("ALTER TABLE ai.naval.foodly_chuck_docs SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

In [0]:
from dotenv import load_dotenv
import os
from typing import Literal, TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.prebuilt.tool_node import ToolNode, tools_condition



from databricks_langchain import (
    ChatDatabricks,
    VectorSearchRetrieverTool
)



# Initialize LLM
llm = ChatDatabricks(endpoint="databricks-gpt-oss-120b")



def call_llm(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state['messages'])]}


# Initialize the retriever tool.
vs_tool = VectorSearchRetrieverTool(
  index_name="ai.naval.foodly_index",
  tool_name="foodly_policy_document_retrieval_tool",
  num_results=2,
  tool_description="Use this tool to search the Foodly knowledge base for policies, procedures, and service-related information. It retrieves the most relevant chunks from the company’s official documentation, including refund rules, cancellation terms, delivery guidelines, loyalty program details, privacy policies, and escalation procedures"
)


llm_with_tools = llm.bind_tools([vs_tool])

builder = StateGraph(MessagesState)

builder.add_node("llm",call_llm)
builder.add_node("tools",ToolNode([vs_tool]))


builder.add_edge(START,"llm")
builder.add_conditional_edges("llm" , tools_condition)
builder.add_edge("tools","llm")


agent = builder.compile()

